# Day 2

Today, we will start using nf-core pipelines to find differentially abundant genes in our dataset. 
We are using data from the following paper: https://www.nature.com/articles/s41593-023-01350-3#Sec10

1. Please take some time to read through the paper and understand their approach, hypotheses and goals.

What was the objective of the study?

> The objective was to look at causes for physical dependence and addiction to opioids in chronic pain therapy on a molecular level. The researches hoped to find a way to treat the withdrawl symptoms to make a switch away from opioids easier for patients. For this they looked at pathways in different brain regions of mice with chronical pain which were put on withdrawl as well as test cohorts. 

**What do the conditions mean?**

oxy: Mice that were given oxycodone, so opioids

sal: Mice that wer given Saline solution, so there wasn't any effect forseeable 

**What do the genotypes mean?**

SNI: Spared nerve injury (SNI), this is a condition where 2/3 branches of the ischias nerve were harmed to put the mouse in chronical pain 

Sham: This is the control group. These mice got an operation, but there nerves weren't damaged in this. Those mice don't have the chronical pain as mice with the SNI genotype.

Imagine you are the bioinformatician in the group who conducted this study. They hand you the raw files and ask you to analyze them.

What would you do?

> I would do:
1. Quality Control (QC) to identify&remove technical/biological bias that may could bias later analysis
2. Normalize the transcriptomics data by doing z-score/log/TPM/CPM normalization
3. Perform differential abundance analysis on samples and do comparison between groups. 

Which groups would you compare to each other?
1. SNI-Oxy with SNI-Sal (and the same for Sham too) to see if there is a difference in expression when oxycodone is administered
2. SNI-Oxy vs Sham-Oxy (and SNI-Sal vs Sham-Sel) to see influence of chronic pain (SNI) to transcription and pathways 
3. SNI-Oxy vs Sham-Sal to see whats the direct difference between a 'normal' mouse and one in chronic pain with an opioid withdrawl

Please also mention which outcome you would expect to see from each comparison.

1. Identify genes/pathways connected to addiction and withdrawl symptoms
2. identify genes that are higher expressed under chronic pain, maybe ones for pain receptors/pain pathways
3. Combination of 1. and 2. may cause some other, previously unknown genes/pathways to be triggered unlike in 1./2.

Your group gave you a very suboptimal excel sheet (conditions_runs_oxy_project.xlsx) to get the information you need for each run they uploaded to the SRA.<br>
So, instead of directly diving into downloading the data and starting the analysis, you first need to sort the lazy table.<br>
Use Python and Pandas to get the table into a more sensible order.<br>
Then, perform some overview analysis and plot the results
1. How many samples do you have per condition?
2. How many samples do you have per genotype?
3. How often do you have each condition per genotype?

In [ ]:
# load excel file into pandas
import pandas as pd

conditions = pd.read_excel("./conditions_runs_oxy_project.xlsx")
conditions = conditions.sort_values(by=["Genotype: SNI", "Condition: Oxy"])

# Convert the marker columns to True/False for easier counting
condition_flags = conditions.eq("x")

In [ ]:
genotype_columns = {
    "SNI": "Genotype: SNI",
    "Sham": "Genotype: Sham",
}
condition_columns = {
    "Oxy": "Condition: Oxy",
    "Sal": "condition: Sal",
}

print("Samples per genotype:")
for genotype, column in genotype_columns.items():
    print(f"{genotype}: {condition_flags[column].sum()}")

print("\nSamples per condition:")
for condition, column in condition_columns.items():
    print(f"{condition}: {condition_flags[column].sum()}")

print("\nConditions per genotype:")
for genotype, genotype_column in genotype_columns.items():
    print(f"{genotype}:")
    for condition, condition_column in condition_columns.items():
        group_samples = condition_flags[genotype_column] & condition_flags[condition_column]
        print(f"  {condition}: {group_samples.sum()}")

> From this I got that there are 8 samples/genotype and 4 per condition and genotype

They were so kind to also provide you with the information of the number of bases per run, so that you can know how much space the data will take on your Cluster.<br>
Add a new column to your fancy table with this information (base_counts.csv) and sort your dataframe according to this information and the condition.

Then select the 2 smallest runs from your dataset and download them from SRA (maybe an nf-core pipeline can help here?...)

In [ ]:
base_counts = pd.read_csv("./base_counts.csv")
conditions = conditions.merge(
    base_counts,
    on="Run",
    how="left",
    validate="one_to_one",
)
conditions = conditions.sort_values(
    by=["Bases", "Condition: Oxy"],
    ascending=[True, True],
)
conditions_reduced = conditions.head(2) # Sorting and picking the head chooses the 2 smallest runs 
conditions_reduced[["Run"]].to_csv("SRA_ids.csv", index=False, header=False) # save to a file required as input by fetchngs

> The 2 smallest runs are `SRR23195516` and `SRR23195511`. To download the files from SRA, I can use the nf-core/fetchngs pipeline [https://nf-co.re/fetchngs/1.13.0/] using the command:

In [ ]:
!nextflow run nf-core/fetchngs \
    -r 1.13.0 \
    -profile docker\
    --input ./SRA_ids.csv \
    --outdir ./fetchngs_out

While your files are downloading, get back to the paper and explain how you would try to reproduce the analysis.<br>
When you are done with this shout, so we can discuss the different ideas.

> To replicate their analysis I would first sort out all the metadata, so check which ref-genome was used and make sure, that I know which conditions relate to which files. Then I would run a pipeline or individual tools to get gene counts, for this I could use nf-core/rnaseq. finally I would do differential expression analysis for the comparisons specified in the paper using DESeq2/the rnaseq pipeline which already includes this step. 